# Mechanistic Hypotheses: Automated Formation & Validation

This notebook demonstrates IRTK's `mechanistic_hypotheses` module for:

1. **Proposing hypotheses** – automatically classifying head roles from patterns
2. **Validating hypotheses** – testing claims via ablation and consistency
3. **Converting to circuits** – mapping hypotheses to circuit graphs
4. **Composing hypotheses** – testing whether components interact
5. **Explaining predictions** – using hypotheses to decompose model outputs

## Why This Matters

Mechanistic hypothesis tools help automate the scientific process of interpretability: forming hypotheses about what circuits do, testing them against model behavior, and refining them based on evidence. This supports scaling interpretability beyond manual analysis.

**Key references:**
- [Wang et al. (2023) "Interpretability in the Wild: IOI"](https://arxiv.org/abs/2211.00593) — Detailed circuit analysis of indirect object identification
- [Conmy et al. (2023) "Towards Automated Circuit Discovery"](https://arxiv.org/abs/2304.14997) — Automated methods for circuit finding via edge pruning

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np

from irtk import HookedTransformer, HookedTransformerConfig
from irtk.mechanistic_hypotheses import (
    MechanisticHypothesis,
    propose_hypotheses,
    validate_hypothesis,
    hypothesis_to_circuit,
    compose_hypotheses,
    explain_prediction,
)

In [ ]:
# Build a small random model
cfg = HookedTransformerConfig(
    n_layers=2, d_model=16, n_ctx=32, d_head=4, n_heads=4, d_vocab=50,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)

seqs = [jnp.array([0, 1, 2, 3, 4, 5]), jnp.array([10, 11, 12, 13, 14, 15])]
print(f"Model: {cfg.n_layers} layers, {cfg.n_heads} heads")

## 1. Propose Hypotheses

Automatically generate hypotheses about what each attention head does by
scoring attention patterns against known roles.

In [ ]:
result = propose_hypotheses(model, seqs)

print(f"Proposed {result['n_hypotheses']} hypotheses")
print(f"Component coverage: {result['component_coverage']:.1%}")
print()
for h in result["hypotheses"]:
    print(f"  {h.component}: {h.role} (confidence={h.confidence:.3f})")
    print(f"    {h.description}")

## 2. Validate a Hypothesis

Test a specific hypothesis via ablation and consistency across inputs.

In [ ]:
# Take a proposed hypothesis (or create one manually)
if result["hypotheses"]:
    hyp = result["hypotheses"][0]
else:
    hyp = MechanisticHypothesis(
        component="L0H0", role="previous_token",
        description="test", hook_name="blocks.0.attn.hook_pattern"
    )

def metric_fn(logits):
    return float(logits[-1, 0])

val = validate_hypothesis(model, hyp, seqs, metric_fn=metric_fn)

print(f"Hypothesis: {hyp.component} -> {hyp.role}")
print(f"Passes: {val['passes']}")
print(f"Consistency: {val['consistency_score']:.3f}")
print(f"Ablation effect: {val['ablation_effect']:.4f}")
print(f"Evidence:")
for e in val["evidence"]:
    print(f"  {e}")

## 3. Hypothesis to Circuit

Convert a hypothesis into a circuit graph with nodes and edges.

In [ ]:
circuit = hypothesis_to_circuit(hyp, model)

print(f"Nodes ({circuit['n_nodes']}): {circuit['nodes']}")
print(f"Edges: {circuit['edges']}")
print(f"Roles:")
for comp, role in circuit["component_roles"].items():
    print(f"  {comp}: {role}")

## 4. Compose Hypotheses

Test whether two hypotheses interact — do they form a compositional circuit?

In [ ]:
h1 = MechanisticHypothesis(
    component="L0H0", role="previous_token",
    description="Previous-token head", hook_name="blocks.0.attn.hook_pattern"
)
h2 = MechanisticHypothesis(
    component="L1H0", role="induction",
    description="Induction head", hook_name="blocks.1.attn.hook_pattern"
)

comp = compose_hypotheses(h1, h2, model, seqs)

print(f"Composes: {comp['composes']}")
print(f"Interaction score: {comp['interaction_score']:.4f}")
print(f"Relationship: {comp['relationship']}")
print(f"h1 effect on h2: {comp['h1_effect_on_h2']:.4f}")

## 5. Explain Prediction

Decompose a prediction into contributions from each hypothesised component.

In [ ]:
hyps = result["hypotheses"][:3] if result["hypotheses"] else [h1, h2]
tokens = jnp.array([0, 1, 2, 3, 4, 5])

expl = explain_prediction(model, tokens, hyps, metric_fn)

print(f"Full metric: {expl['full_metric']:.4f}")
print(f"Total explained: {expl['total_explained']:.4f}")
print(f"\nComponent effects (by importance):")
for comp in expl["explanation_order"]:
    print(f"  {comp}: effect = {expl['component_effects'][comp]:.4f}")